1. Importing Necessary Libraries

In [1]:
import os, glob
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras
from PIL import Image, ImageOps
import cv2
import pandas as pd
from tqdm import tqdm

print("TF:", tf.__version__)


TF: 2.15.1


2. Global configuration for ANN/CNN/RNN adversarial experiments

In [2]:
# Dataset & model paths 
BASE_DIR        = '.'                                    # Project root; used to resolve relative image paths
CSV_PATH        = '../../annotations/new_annotations.csv'                # Annotations CSV with columns: image_id, label, isNight and split

ANN_MODEL_PATH  = '../../models/traffic_light_ann.keras'     # Feed-forward ANN (.keras)
CNN_MODEL_PATH  = '../../models/traffic_light_cnn.keras'     # CNN backbone (.keras); may be single- or multi-input
RNN_MODEL_PATH  = '../../models/traffic_light_rnn.keras'     # RNN (.keras); typically expects (50,150) sequences

# Image preprocessing
IMG_SIZE        = 50     # Target spatial size (50x50); matches backend preprocess_50x50

# Attack hyperparameters 
EPS_FGSM        = 0.10              # FGSM L_inf budget (in input units, e.g., 0–1 range if inputs are normalized)
EPS_PGD         = 0.10              # PGD L_inf budget (max per-pixel perturbation)
PGD_STEPS       = 40                # Number of PGD iterations
PGD_STEP_SIZE   = EPS_PGD / 10.0    # PGD step size per iteration (commonly eps/4 ~ eps/10)

# Loss configuration 
# Set to True if the model outputs logits (i.e., last layer has linear activation/no softmax).
# Set to False if the model outputs probabilities (softmax applied in the model).
ANN_USE_LOGITS  = False
CNN_USE_LOGITS  = False
RNN_USE_LOGITS  = False

# Output directories 
SAVE_DIR_ROOT   = '../../datasets/adv_outputs'               # Root folder where all adversarial PNGs will be saved
os.makedirs(SAVE_DIR_ROOT, exist_ok=True)       # Create the directory tree if it doesn't exist

3. Robust CSV loading and path utilities (OS-agnostic, mirror-save helpers)

In [3]:
# CSV + path utils 
def read_csv_robust(path):
    """
    Read a CSV file using a set of fallback separators/encodings.

    Tries multiple (separator, encoding) combinations until reading succeeds
    and the dataframe has at least 2 columns (required for image_id + label).

    Args:
        path: Filesystem path to the CSV.

    Returns:
        pandas.DataFrame: The parsed table.

    Raises:
        AssertionError: If the CSV cannot be read with any of the tried combinations.
    """
    for sep in [None, ',', ';', '\t']:
        for enc in ['utf-8-sig', 'utf-8', 'cp1252']:
            try:
                df = pd.read_csv(path, sep=sep, engine='python', encoding=enc)
                # Require at least two columns (e.g., image_id and label).
                if df.shape[1] >= 2:
                    return df
            except Exception:
                # Swallow and continue trying the next (sep, enc) pair.
                pass
    raise AssertionError(f'Failed to read CSV: {path}')

def resolve_path(image_id):
    """
    Normalize a potentially relative image path against BASE_DIR.

    Converts forward/back slashes to the current OS separator and returns
    an absolute or normalized relative path.

    Args:
        image_id: Path-like value (relative or absolute) coming from the CSV.

    Returns:
        str: A normalized filesystem path suitable for reading the image.
    """
    # Normalize slashes for cross-platform consistency.
    p = str(image_id).replace('\\', os.sep).replace('/', os.sep)
    # If already absolute, return as-is; otherwise, resolve against BASE_DIR.
    if os.path.isabs(p): return p
    return os.path.normpath(os.path.join(BASE_DIR, p))




def mirror_out_path(save_root, original_path):
    p = str(original_path).replace('\\', os.sep).replace('/', os.sep)

    parts = p.split(os.sep)

    if 'processed_images' in parts:
        idx = parts.index('processed_images')
        rel = os.path.join(*parts[idx:])   # processed_images/test/img_0.jpg
    else:
        rel = os.path.basename(p)

    out_path = os.path.join(save_root, os.path.splitext(rel)[0] + '.png')
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    return out_path

4. Image preprocessing utilities (RGB normalized & backend-equivalent BGR 50×50)

In [4]:
# Preprocessing 
def preprocess_rgb(path, target=IMG_SIZE):
    """
    Load an image as RGB, apply EXIF-aware orientation, resize to `target`×`target`,
    and normalize to [0, 1].

    Args:
        path: Filesystem path to the input image.
        target: Output spatial size (width = height = target).

    Returns:
        np.ndarray: Float32 RGB array with shape (target, target, 3) in [0, 1].
    """
    with Image.open(path) as pil:
        # Respect EXIF orientation (rotations/mirroring) and ensure RGB mode.
        pil = ImageOps.exif_transpose(pil.convert('RGB'))
        arr = np.array(pil)  # uint8 RGB

    # Area (box) interpolation is good for downscaling natural images.
    arr = cv2.resize(arr, (target, target), interpolation=cv2.INTER_AREA)

    # Normalize to [0,1] float32 for model consumption.
    return (arr.astype(np.float32) / 255.0)

def preprocess_bgr_50x50(path_or_stream):
    """
    Load an image and return a backend-equivalent BGR tensor:
    EXIF-aware orientation → RGB → BGR → resize to 50×50 → normalize to [0, 1].

    Accepts either a filesystem path (str/Path) or a file-like object (e.g., Flask
    `werkzeug.datastructures.FileStorage.stream`).

    Args:
        path_or_stream: Image path or a readable file-like stream.

    Returns:
        np.ndarray: Float32 BGR array with shape (50, 50, 3) in [0, 1].
    """
    # Decide if we got a file-like object (has .read) or a path.
    if hasattr(path_or_stream, 'read'):
        pil = Image.open(path_or_stream).convert('RGB')
        pil = ImageOps.exif_transpose(pil)
        arr = np.array(pil)
    else:
        with Image.open(path_or_stream) as pil:
            pil = ImageOps.exif_transpose(pil.convert('RGB'))
            arr = np.array(pil)

    # Convert channel order to BGR to match backend expectations.
    bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)

    # Resize to fixed 50×50 (aligned with backend preprocess_50x50).
    bgr = cv2.resize(bgr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    # Normalize to [0,1] float32.
    return (bgr.astype(np.float32) / 255.0)

5. FGSM & PGD for ANN with fixed auxiliary feature (isNight)

In [5]:
# ANN attacks (vector: flatten(BGR)+isNight) 
def fgsm_ann(model, x_input_np: np.ndarray, eps: float = 0.1, from_logits: bool = False) -> np.ndarray:
    """
    Untargeted FGSM on an ANN that takes a flattened image plus an auxiliary feature.

    The input vector is assumed to be:
        [flattened BGR pixels (H*W*3), isNight (last element)]
    We perturb only the pixel part and keep the last feature (isNight) unchanged.

    Args:
        model: Keras model mapping (N, D) -> (N, C).
        x_input_np: Input batch as float32, shape (N, H*W*3 + 1).
        eps: Linf budget for FGSM (in input units).
        from_logits: Set True if model outputs logits (no softmax inside the model).

    Returns:
        np.ndarray with same shape as x_input_np containing adversarial examples.
    """
    x = tf.convert_to_tensor(x_input_np, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x)
        preds = model(x, training=False)
        # Use model's own prediction as label (standard untargeted FGSM)
        y_idx = tf.argmax(preds, axis=1)
        y_true = tf.one_hot(y_idx, depth=int(preds.shape[-1]))
        loss = tf.keras.losses.categorical_crossentropy(y_true, preds, from_logits=from_logits)

    grad = tape.gradient(loss, x)
    signed = tf.sign(grad)

    # Mask to prevent changing the last feature (isNight)
    mask = tf.concat([tf.ones_like(x[:, :-1]), tf.zeros_like(x[:, -1:])], axis=1)  # keep isNight fixed
    signed = signed * mask

    x_adv = x + eps * signed
    x_adv_np = x_adv.numpy()

    # Explicitly preserve isNight and clip pixels to [0, 1]
    x_adv_np[:, -1] = x_input_np[:, -1]
    x_adv_np[:, :-1] = np.clip(x_adv_np[:, :-1], 0.0, 1.0)
    return x_adv_np

def pgd_ann(model, x_input_np: np.ndarray, eps: float = 0.1, step: float = 0.01, steps: int = 40, from_logits: bool=False) -> np.ndarray:
    """
    Untargeted PGD on an ANN with input [flattened pixels..., isNight].

    Projected Gradient Descent with Linf constraint:
    - Iteratively takes FGSM-like steps of size `step`
    - Projects the perturbation back to the Linf ball of radius `eps`
    - Does not modify the last feature (isNight)
    - Clips pixel features to [0, 1]

    Args:
        model: Keras model mapping (N, D) -> (N, C).
        x_input_np: Input batch as float32, shape (N, H*W*3 + 1).
        eps: Linf radius for the perturbation.
        step: Per-iteration step size.
        steps: Number of PGD iterations.
        from_logits: Set True if model outputs logits.

    Returns:
        np.ndarray adversarial batch with same shape as x_input_np.
    """
    x0 = tf.convert_to_tensor(x_input_np, dtype=tf.float32)
    x = tf.identity(x0)
    for _ in range(int(steps)):
        with tf.GradientTape() as tape:
            tape.watch(x)
            preds = model(x, training=False)
            y_idx = tf.argmax(preds, axis=1)
            y_true = tf.one_hot(y_idx, depth=int(preds.shape[-1]))
            loss = tf.keras.losses.categorical_crossentropy(y_true, preds, from_logits=from_logits)

        grad = tape.gradient(loss, x)
        signed = tf.sign(grad)

        # Zero-out the gradient on isNight (last feature)
        mask = tf.concat([tf.ones_like(x[:, :-1]), tf.zeros_like(x[:, -1:])], axis=1)
        signed = signed * mask

        # Gradient ascent step on pixels
        x = x + step * signed

        # Project pixel perturbation into Linf-ball and keep isNight intact
        delta = x - x0
        delta_pixels = tf.clip_by_value(delta[:, :-1], -eps, eps)
        x = tf.concat([x0[:, :-1] + delta_pixels, x0[:, -1:]], axis=1)

        # Clip pixel values to [0,1]
        x_pixels = tf.clip_by_value(x[:, :-1], 0.0, 1.0)
        x = tf.concat([x_pixels, x[:, -1:]], axis=1)

    return x.numpy()

def vector_to_rgb_image(x_row, img_size=IMG_SIZE):
    """
    Convert a single ANN input vector back to an RGB image (uint8).

    Assumes `x_row` is:
        [flattened BGR pixels (img_size*img_size*3), isNight]
    The function ignores the final element and reconstructs the image.

    Args:
        x_row: 1D array containing flattened BGR pixels followed by isNight.
        img_size: Spatial size used for the original flattening.

    Returns:
        np.ndarray uint8 RGB image with shape (img_size, img_size, 3).
    """
    bgr = x_row[:-1].reshape(img_size, img_size, 3)
    rgb = cv2.cvtColor((bgr * 255).astype('uint8'), cv2.COLOR_BGR2RGB)
    return rgb

6. Pixel-space FGSM & PGD (single-/multi-input) with 𝐿∞ constraint

In [6]:
# Pixel-space attacks (single & multi) 
def fgsm_pixels(model, x_pixels, eps=0.1, from_logits=False, y_true=None):
    """
    Untargeted FGSM on image pixels (single-input model).

    Args:
        model: Keras/TensorFlow model taking images only (N,H,W,3).
        x_pixels: Float32 tensor/ndarray of shape (N,H,W,3), values in [0,1].
        eps: Linf budget; max absolute change per pixel channel.
        from_logits: True if `model` returns logits (no softmax).
        y_true: Optional one-hot labels of shape (N,C). If None, uses model
                predictions as labels (standard untargeted FGSM).

    Returns:
        np.ndarray adversarial examples with same shape as x_pixels.
    """
    x0 = tf.convert_to_tensor(x_pixels, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x0)
        preds = model(x0, training=False)
        if y_true is None:
            y_idx = tf.argmax(preds, axis=1)
            y_true = tf.one_hot(y_idx, depth=int(preds.shape[-1]))
        loss = tf.keras.losses.categorical_crossentropy(y_true, preds, from_logits=from_logits)
    grad = tape.gradient(loss, x0)
    x_adv = x0 + eps * tf.sign(grad)                # one-step Linf ascent
    x_adv = tf.clip_by_value(x_adv, 0.0, 1.0)       # one-step Linf ascent
    return x_adv.numpy()

def pgd_pixels(model, x_pixels, eps=0.1, step=0.01, steps=40, from_logits=False, y_true=None):
    """
    Untargeted PGD on image pixels (single-input model) with Linf projection.

    Args:
        model: Keras/TensorFlow model taking images only (N,H,W,3).
        x_pixels: Float32 (N,H,W,3) in [0,1].
        eps: Linf radius (max perturbation per channel).
        step: Per-iteration Linf step size.
        steps: Number of iterations.
        from_logits: True if `model` returns logits (no softmax).
        y_true: Optional one-hot labels (N,C); if None, uses current preds.

    Returns:
        np.ndarray adversarial examples with same shape as x_pixels.
    """
    x0 = tf.convert_to_tensor(x_pixels, dtype=tf.float32)
    x = tf.identity(x0)
    for _ in range(int(steps)):
        with tf.GradientTape() as tape:
            tape.watch(x)
            preds = model(x, training=False)
            if y_true is None:
                y_idx = tf.argmax(preds, axis=1)
                y_true_dyn = tf.one_hot(y_idx, depth=int(preds.shape[-1]))
            else:
                y_true_dyn = y_true
            loss = tf.keras.losses.categorical_crossentropy(y_true_dyn, preds, from_logits=from_logits)
        grad = tape.gradient(loss, x)
        x = x + step * tf.sign(grad)    
        x = tf.clip_by_value(tf.minimum(x0 + eps, tf.maximum(x0 - eps, x)), 0.0, 1.0)
    return x.numpy()

def fgsm_pixels_multi(model, x_pixels, aux, eps=0.1, from_logits=False, y_true=None):
    """
    Untargeted FGSM on image pixels for multi-input models (e.g., [image, isNight]).

    Only the image tensor is perturbed; the auxiliary input `aux` is kept fixed.

    Args:
        model: Keras/TensorFlow model taking [images, aux].
        x_pixels: Float32 (N,H,W,3) in [0,1].
        aux: Float32 auxiliary batch (N, A), passed through unchanged.
        eps: Linf budget on pixels.
        from_logits: True if `model` returns logits (no softmax).
        y_true: Optional one-hot labels (N,C); if None, uses current preds.

    Returns:
        np.ndarray adversarial images with same shape as x_pixels.
    """
    x0 = tf.convert_to_tensor(x_pixels, dtype=tf.float32)
    aux_tf = tf.convert_to_tensor(aux, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x0)
        preds = model([x0, aux_tf], training=False)
        if y_true is None:
            y_idx = tf.argmax(preds, axis=1)
            y_true = tf.one_hot(y_idx, depth=int(preds.shape[-1]))
        loss = tf.keras.losses.categorical_crossentropy(y_true, preds, from_logits=from_logits)
    grad = tape.gradient(loss, x0)
    x_adv = x0 + eps * tf.sign(grad)                # perturb only pixels
    x_adv = tf.clip_by_value(x_adv, 0.0, 1.0)
    return x_adv.numpy()

def pgd_pixels_multi(model, x_pixels, aux, eps=0.1, step=0.01, steps=40, from_logits=False, y_true=None):
    """
    Untargeted PGD on image pixels for multi-input models (e.g., [image, isNight]).

    Perturbs only the image tensor; auxiliary input `aux` remains unchanged.

    Args:
        model: Keras/TensorFlow model taking [images, aux].
        x_pixels: Float32 (N,H,W,3) in [0,1].
        aux: Float32 auxiliary batch (N, A), passed through as-is.
        eps: Linf radius on pixels.
        step: Per-iteration step size.
        steps: Number of iterations.
        from_logits: True if `model` returns logits (no softmax).
        y_true: Optional one-hot labels (N,C); if None, uses current preds.

    Returns:
        np.ndarray adversarial images with same shape as x_pixels.
    """
    x0 = tf.convert_to_tensor(x_pixels, dtype=tf.float32)
    aux_tf = tf.convert_to_tensor(aux, dtype=tf.float32)
    x = tf.identity(x0)
    for _ in range(int(steps)):
        with tf.GradientTape() as tape:
            tape.watch(x)
            preds = model([x, aux_tf], training=False)
            if y_true is None:
                y_idx = tf.argmax(preds, axis=1)
                y_true_dyn = tf.one_hot(y_idx, depth=int(preds.shape[-1]))
            else:
                y_true_dyn = y_true
            loss = tf.keras.losses.categorical_crossentropy(y_true_dyn, preds, from_logits=from_logits)
        grad = tape.gradient(loss, x)
        x = x + step * tf.sign(grad)
        x = tf.clip_by_value(tf.minimum(x0 + eps, tf.maximum(x0 - eps, x)), 0.0, 1.0)
    return x.numpy()

7. CNN BGR wrappers (backend-equivalent; optional aux input)

In [7]:
# CNN wrappers (BGR like backend)
def build_cnn_bgr_wrapper_single(base_model, img_size=IMG_SIZE):
    """
    Wrap a single-input CNN so it accepts BGR images of shape (IMG_SIZE, IMG_SIZE, 3).

    The wrapper:
      - takes (H, W, 3) BGR in [0,1],
      - resizes to the base model's expected size if needed,
      - forwards the tensor directly (no channel reordering).

    Args:
        base_model: The original CNN model (expects an image tensor).
        img_size:   The incoming image size used in preprocessing (e.g., 50).

    Returns:
        A Keras Model that can be called with BGR images of shape (None, IMG_SIZE, IMG_SIZE, 3).
    """
    H=W=img_size
    pix_in = keras.Input(shape=(H,W,3), name='bgr_pixels')
    x = pix_in
    shp = base_model.input_shape
    in_shape = shp if isinstance(shp, (list,tuple)) else (None,) + tuple(shp[1:])
    if len(in_shape)>=4 and in_shape[1] not in (None,-1) and in_shape[2] not in (None,-1):
        tH, tW = int(in_shape[1]), int(in_shape[2])
        if (tH,tW)!=(H,W):
            x = keras.layers.Lambda(lambda t: tf.image.resize(t, (tH,tW)))(x)
    out = base_model(x)  # pass BGR as-is
    return keras.Model(pix_in, out, name=f'cnn_bgr_wrapper_{base_model.name}')

def build_cnn_bgr_wrapper_multi(base_model, img_size=IMG_SIZE, aux_dim=1):
    """
    Wrap a multi-input CNN so it accepts:
      - BGR images of shape (IMG_SIZE, IMG_SIZE, 3), and
      - an auxiliary vector (e.g., [isNight]) of shape (aux_dim,).

    The wrapper:
      - resizes the image to the base model's expected size if needed,
      - forwards both inputs to the base model without altering their semantics.

    Args:
        base_model: The original CNN model (expects [image_tensor, aux_tensor]).
        img_size:   The incoming image size used in preprocessing (e.g., 50).
        aux_dim:    Dimensionality of the auxiliary input (default 1 for isNight).

    Returns:
        A Keras Model that can be called with [BGR image, aux] inputs.
    """
    H=W=img_size
    pix_in = keras.Input(shape=(H,W,3), name='bgr_pixels')
    aux_in = keras.Input(shape=(aux_dim,), name='aux_input')
    x = pix_in
    shp = base_model.input_shape
    in_shape = shp if isinstance(shp, (list,tuple)) else (None,) + tuple(shp[1:])
    img_shape = in_shape[0] if isinstance(in_shape, (list,tuple)) else in_shape
    if len(img_shape)>=4 and img_shape[1] not in (None,-1) and img_shape[2] not in (None,-1):
        tH, tW = int(img_shape[1]), int(img_shape[2])
        if (tH,tW)!=(H,W):
            x = keras.layers.Lambda(lambda t: tf.image.resize(t, (tH,tW)))(x)
    out = base_model([x, aux_in])  # pass BGR as-is
    return keras.Model([pix_in, aux_in], out, name=f'cnn_bgr_wrapper_multi_{base_model.name}')

8. RNN BGR wrappers (reshape 50×50×3 → sequence 50×150, optional aux input)

In [8]:
# RNN wrappers (BGR 50x50 -> reshape to (50,150)) 
def build_rnn_bgr_wrapper_single(base_model, img_size=IMG_SIZE):
    """
    Wrap an RNN that expects a (T=50, F=150) sequence to accept BGR images.

    Converts an input BGR image (H=W=img_size, C=3) to a sequence by
    reshaping (H, W, 3) → (T=H, F=W*3). For IMG_SIZE=50 this becomes (50, 150).

    Args:
        base_model: RNN base that takes sequences of shape (None, 50, 150).
        img_size:   Spatial size of the incoming image (default 50).

    Returns:
        A Keras Model that accepts (None, img_size, img_size, 3) BGR and forwards
        the reshaped sequence to the base RNN.
    """
    H=W=img_size
    pix_in = keras.Input(shape=(H,W,3), name='bgr_pixels')
    # Reshape pixel grid into a time–feature sequence: (N, H, W*3)
    seq = keras.layers.Lambda(lambda t: tf.reshape(t, (-1, H, W*3)))(pix_in)  
    out = base_model(seq)
    return keras.Model(pix_in, out, name='rnn_bgr_wrapper_single')

def build_rnn_bgr_wrapper_multi(base_model, img_size=IMG_SIZE, aux_dim=1):
    """
    Same as the single-input wrapper but supports an auxiliary vector (e.g., [isNight]).

    Inputs:
      - BGR image (H=W=img_size, C=3), normalized to [0,1]
      - Auxiliary vector of shape (aux_dim,) passed unchanged to the base model

    The BGR image is reshaped to a sequence (H, W*3) == (50,150) for IMG_SIZE=50.

    Args:
        base_model: RNN base that expects [sequence, aux] inputs.
        img_size:   Spatial size of the incoming image (default 50).
        aux_dim:    Dimension of the auxiliary input (default 1).

    Returns:
        A Keras Model that takes [BGR image, aux] and forwards
        [reshaped sequence, aux] to the base RNN.
    """
    H=W=img_size
    pix_in = keras.Input(shape=(H,W,3), name='bgr_pixels')
    aux_in = keras.Input(shape=(aux_dim,), name='aux_input')

    # Reshape pixel grid into (N, H, W*3) sequence
    seq = keras.layers.Lambda(lambda t: tf.reshape(t, (-1, H, W*3)))(pix_in)
    out = base_model([seq, aux_in])
    return keras.Model([pix_in, aux_in], out, name='rnn_bgr_wrapper_multi')

9. Build dataset items from CSV (paths, labels, optional isNight)

In [9]:
# Build items 
# Infer column names (case-insensitive) and assemble a list of (path, label, isNight)
df = read_csv_robust(CSV_PATH)

# Map lowercase->original column name to allow case-insensitive lookups.
cols = {c.lower(): c for c in df.columns}

# Column selection (fallbacks):
# - image: prefer 'image_id', then 'image', else first column
# - label: prefer 'label', else second column
# - isnight: optional (defaults to 0.0 if missing)
col_image = cols.get('image_id') or cols.get('image') or list(df.columns)[0]
col_label = cols.get('label') or list(df.columns)[1]
col_night = cols.get('isnight')

# Resolve image paths against BASE_DIR for cross-platform execution.
paths  = [resolve_path(p) for p in df[col_image].tolist()]

# Ensure correct dtypes for downstream code (int labels, float isNight).
labels = df[col_label].astype(int).tolist()
nights = df[col_night].astype(float).tolist() if col_night is not None else [0.0]*len(paths)

# Final dataset tuples: (absolute_or_normalized_path, label_int, isNight_float)
items = list(zip(paths, labels, nights))

print(f"Loaded {len(items)} items. Example:", items[:2])

Loaded 51826 items. Example: [('..\\..\\datasets\\processed_images\\train\\img_0.jpg', 1, 0.0), ('..\\..\\datasets\\processed_images\\train\\img_1.jpg', 1, 0.0)]


10. Load models and construct wrappers (ANN / CNN / RNN)

In [10]:
# Load models & build wrappers 
models = {}

# ANN
if os.path.exists(ANN_MODEL_PATH):
    # Load the feed-forward ANN (expects flattened pixels + isNight)
    models['ANN'] = keras.models.load_model(ANN_MODEL_PATH)
    print("Loaded ANN:", models['ANN'].input_shape, "->", models['ANN'].output_shape)
else:
    print("[WARN] ANN model not found:", ANN_MODEL_PATH)

# CNN (BGR like backend)
if os.path.exists(CNN_MODEL_PATH):
    # Load the base CNN; can be single-input (image) or multi-input ([image, aux])
    models['CNN_base'] = keras.models.load_model(CNN_MODEL_PATH)
    print("Loaded CNN base:", models['CNN_base'].input_shape, "->", models['CNN_base'].output_shape)

    base_in_shape = models['CNN_base'].input_shape
    # Heuristic: if input_shape is a list/tuple of length >= 2, treat as multi-input and
    # infer aux feature dimension from the second input's last axis.
    if isinstance(base_in_shape, (list, tuple)) and len(base_in_shape) >= 2:
        aux_shape = base_in_shape[1]        # e.g., 1 for isNight
        aux_dim = 1
        try:
            if aux_shape is not None and hasattr(aux_shape, '__len__') and len(aux_shape) > 0 and aux_shape[-1] not in (None, -1):
                aux_dim = int(aux_shape[-1])
        except Exception:
            aux_dim = 1

        # Build multi-input wrapper: passes BGR image + aux to the base CNN (auto-resize if needed)
        models['CNN'] = build_cnn_bgr_wrapper_multi(models['CNN_base'], IMG_SIZE, aux_dim=aux_dim)
        models['CNN_needs_aux'] = True
        print("Built CNN BGR multi wrapper; aux_dim=", aux_dim)
    else:
        # Single-input wrapper: passes only BGR image (auto-resize if needed)
        models['CNN'] = build_cnn_bgr_wrapper_single(models['CNN_base'], IMG_SIZE)
        models['CNN_needs_aux'] = False
        print("Built CNN BGR single wrapper")
else:
    print("[WARN] CNN model not found:", CNN_MODEL_PATH)

# RNN (BGR reshape like backend)
if os.path.exists(RNN_MODEL_PATH):
    # Load the base RNN; can be single-input (sequence) or multi-input ([sequence, aux])
    models['RNN_base'] = keras.models.load_model(RNN_MODEL_PATH)
    print("Loaded RNN base:", models['RNN_base'].input_shape, "->", models['RNN_base'].output_shape)

    base_in_shape = models['RNN_base'].input_shape
    # Heuristic: if input_shape is list/tuple of length >= 2, treat as multi-input
    if isinstance(base_in_shape, (list, tuple)) and len(base_in_shape) >= 2:
        aux_shape = base_in_shape[1]
        aux_dim = 1
        try:
            if aux_shape is not None and hasattr(aux_shape, '__len__') and len(aux_shape) > 0 and aux_shape[-1] not in (None, -1):
                aux_dim = int(aux_shape[-1])
        except Exception:
            aux_dim = 1

        # Multi-input wrapper: BGR (50×50×3) → reshape to (50,150) sequence, plus aux passthrough
        models['RNN'] = build_rnn_bgr_wrapper_multi(models['RNN_base'], IMG_SIZE, aux_dim=aux_dim)
        models['RNN_needs_aux'] = True
        print("Built RNN BGR multi wrapper; aux_dim=", aux_dim)
    else:
        # Single-input wrapper: only sequence derived from BGR image
        models['RNN'] = build_rnn_bgr_wrapper_single(models['RNN_base'], IMG_SIZE)
        models['RNN_needs_aux'] = False
        print("Built RNN BGR single wrapper")
else:
    print("[WARN] RNN model not found:", RNN_MODEL_PATH)

[WARN] ANN model not found: ../../models/traffic_light_ann.keras



Loaded CNN base: [(None, 50, 50, 3), (None, 1)] -> (None, 3)
Built CNN BGR multi wrapper; aux_dim= 1
Loaded RNN base: [(None, 50, 150), (None, 1)] -> (None, 3)
Built RNN BGR multi wrapper; aux_dim= 1


11. Run ANN attacks over dataset and save adversarial PNGs

In [11]:
# ANN attacks
# Runs FGSM and PGD for the ANN model on every (path, label, isNight) item:
# Input vector = flatten(BGR 50x50) + isNight (last feature)
# Perturbations are applied ONLY on pixel features; isNight stays fixed
# Saves adversarial images (reconstructed RGB PNG) mirroring original paths

if 'ANN' in models:
    ann = models['ANN']

    # Output roots for adversarial images
    out_root_fgsm = os.path.join(SAVE_DIR_ROOT, 'ANN_fgsm')
    out_root_pgd  = os.path.join(SAVE_DIR_ROOT, 'ANN_pgd')
    os.makedirs(out_root_fgsm, exist_ok=True)
    os.makedirs(out_root_pgd,  exist_ok=True)

    # Simple running metrics
    clean_correct = adv_fgsm_correct = adv_pgd_correct = total = 0

    for path, label, is_night in tqdm(items, desc='ANN attacks', unit='img'):
        # Load image as BGR [0,1], 50x50 (backend-equivalent)
        try:
            bgr = preprocess_bgr_50x50(path)
        except Exception as e:
            print("[WARN] cannot load:", path, "->", e)
            continue

        # Build ANN input: flatten pixels + isNight
        x_flat = bgr.reshape(1, -1)                                         # shape (1, H*W*3)
        night  = np.array([[float(is_night)]], dtype=np.float32)            # shape (1, 1)
        x      = np.hstack([x_flat, night]).astype(np.float32)              # shape (1, H*W*3 + 1)

        # Clean prediction (baseline) 
        clean_idx = int(np.argmax(ann.predict(x, verbose=0)[0]))
        clean_correct += int(clean_idx == label)


        # FGSM (untargeted) on vector space; keep isNight intact 
        x_adv_fgsm = fgsm_ann(ann, x, eps=EPS_FGSM, from_logits=ANN_USE_LOGITS)

        # Save adversarial PNG: reconstruct pixels (BGR->RGB) and mirror path
        Image.fromarray(vector_to_rgb_image(x_adv_fgsm[0], IMG_SIZE)).save(mirror_out_path(out_root_fgsm, path))

        # Evaluate on adversarial example
        idx_fgsm = int(np.argmax(ann.predict(x_adv_fgsm, verbose=0)[0]))
        adv_fgsm_correct += int(idx_fgsm == label)

        # PGD (untargeted) with Linf projection; isNight fixed ---
        x_adv_pgd = pgd_ann(ann, x, eps=EPS_PGD, step=PGD_STEP_SIZE, steps=PGD_STEPS, from_logits=ANN_USE_LOGITS)

        # Save PGD adversarial PNG
        Image.fromarray(vector_to_rgb_image(x_adv_pgd[0], IMG_SIZE)).save(mirror_out_path(out_root_pgd, path))

        # Evaluate on PGD adversarial
        idx_pgd = int(np.argmax(ann.predict(x_adv_pgd, verbose=0)[0]))
        adv_pgd_correct += int(idx_pgd == label)

        total += 1

    # Print simple accuracy summary
    print(f"[ANN] clean_acc={clean_correct/max(1,total):.4f}  fgsm_acc={adv_fgsm_correct/max(1,total):.4f}  pgd_acc={adv_pgd_correct/max(1,total):.4f}")
else:
    print("Skipping ANN.")

Skipping ANN.


12. Run CNN attacks over dataset and save adversarial PNGs

In [ ]:
# CNN attacks 
# Applies untargeted FGSM and PGD directly on pixel inputs in BGR [0,1] space.
# If the underlying CNN is multi-input ([image, aux]), `isNight` is passed but never perturbed.
# Saves adversarial images as PNG (BGR→RGB) mirroring the original folder structure.

if 'CNN' in models:
    cnn = models['CNN']

    # Output folders for adversarial images produced by FGSM and PGD
    out_root_fgsm = os.path.join(SAVE_DIR_ROOT, 'CNN_fgsm')
    out_root_pgd  = os.path.join(SAVE_DIR_ROOT, 'CNN_pgd')
    os.makedirs(out_root_fgsm, exist_ok=True)
    os.makedirs(out_root_pgd,  exist_ok=True)

    # Simple accuracy counters
    clean_correct = adv_fgsm_correct = adv_pgd_correct = total = 0

    # Iterate over all dataset items: (path, label, isNight)
    for path, label, is_night in tqdm(items, desc='CNN attacks (backend-equiv BGR)', unit='img'):
        try:
            # Load as BGR [0,1], resize to 50x50 (backend-equivalent preprocessing)
            bgr = preprocess_bgr_50x50(path)  
        except Exception as e:
            print("[WARN] cannot load:", path, "->", e)
            continue

        x = bgr[np.newaxis, ...]    # model expects batch dimension
        needs_aux = bool(models.get('CNN_needs_aux'))
        if needs_aux:
            # Auxiliary input (e.g., [isNight]) flows through unchanged
            aux = np.array([[float(is_night)]], dtype=np.float32)

        # Clean prediction (baseline) 
        if needs_aux:
            clean_pred = cnn.predict([x, aux], verbose=0)[0]
        else:
            clean_pred = cnn.predict(x, verbose=0)[0]
        clean_idx = int(np.argmax(clean_pred))
        clean_correct += int(clean_idx == label)

        # FGSM (untargeted) on pixels only 
        if needs_aux:
            x_adv_fgsm = fgsm_pixels_multi(cnn, x, aux, eps=EPS_FGSM, from_logits=CNN_USE_LOGITS)
            adv_pred_fgsm = cnn.predict([x_adv_fgsm, aux], verbose=0)[0]
        else:
            x_adv_fgsm = fgsm_pixels(cnn, x, eps=EPS_FGSM, from_logits=CNN_USE_LOGITS)
            adv_pred_fgsm = cnn.predict(x_adv_fgsm, verbose=0)[0]

        # Save FGSM image as RGB PNG (convert from BGR)
        rgb_png = cv2.cvtColor((x_adv_fgsm[0]*255).astype('uint8'), cv2.COLOR_BGR2RGB)
        Image.fromarray(rgb_png).save(mirror_out_path(out_root_fgsm, path))

        idx_fgsm = int(np.argmax(adv_pred_fgsm))
        adv_fgsm_correct += int(idx_fgsm == label)

        # PGD (untargeted) on pixels only 
        if needs_aux:
            x_adv_pgd = pgd_pixels_multi(cnn, x, aux, eps=EPS_PGD, step=PGD_STEP_SIZE, steps=PGD_STEPS, from_logits=CNN_USE_LOGITS)
            adv_pred_pgd = cnn.predict([x_adv_pgd, aux], verbose=0)[0]
        else:
            x_adv_pgd = pgd_pixels(cnn, x, eps=EPS_PGD, step=PGD_STEP_SIZE, steps=PGD_STEPS, from_logits=CNN_USE_LOGITS)
            adv_pred_pgd = cnn.predict(x_adv_pgd, verbose=0)[0]

            
        # Save PGD image as RGB PNG (convert from BGR)
        rgb_png = cv2.cvtColor((x_adv_pgd[0]*255).astype('uint8'), cv2.COLOR_BGR2RGB)
        Image.fromarray(rgb_png).save(mirror_out_path(out_root_pgd, path))

        idx_pgd = int(np.argmax(adv_pred_pgd))
        adv_pgd_correct += int(idx_pgd == label)

        total += 1

    # Accuracy summary (clean vs adversarial)
    print(f"[CNN] clean_acc={clean_correct/max(1,total):.4f}  fgsm_acc={adv_fgsm_correct/max(1,total):.4f}  pgd_acc={adv_pgd_correct/max(1,total):.4f}")
else:
    print("Skipping CNN.")

CNN attacks (backend-equiv BGR):   0%|          | 54/51826 [00:46<12:07:07,  1.19img/s]

13. Generate adversarials only for test set

In [ ]:
# Build a list of (path, label, isNight) tuples for samples that belong to split == 'test'

test_rows = df[df['split']=='test'][['image_id','label','isNight']].dropna()
items_test = [(str(r['image_id']), int(r['label']), int(r['isNight'])) for _, r in test_rows.iterrows()]

print(f"Will generate adversarials for {len(items_test)} test images.")
# After this, replace:
# for path, label, is_night in tqdm(items, ...):
# with:
# for path, label, is_night in tqdm(items_test, ...):
# inside your adversarial-generation block.


Will generate adversarials for 14433 test images.


14. Run RNN attacks over dataset and save adversarial PNGs

In [ ]:
# RNN attacks (BGR like backend; reshape to (50,150)) 
# Applies untargeted FGSM and PGD directly on image pixels in BGR [0,1].
# The RNN wrapper reshapes BGR (50,50,3) -> sequence (50,150), mirroring the backend.
# If the underlying RNN is multi-input ([sequence, aux]), `isNight` is passed but never perturbed.
# Adversarial images are saved as PNG (BGR→RGB) while preserving directory structure.
if 'RNN' in models:
    rnn = models['RNN']

    # Output folders for adversarial images
    out_root_fgsm = os.path.join(SAVE_DIR_ROOT, 'RNN_fgsm')
    out_root_pgd  = os.path.join(SAVE_DIR_ROOT, 'RNN_pgd')
    os.makedirs(out_root_fgsm, exist_ok=True)
    os.makedirs(out_root_pgd,  exist_ok=True)

    # Simple running metrics
    clean_correct = adv_fgsm_correct = adv_pgd_correct = total = 0

    # Iterate over dataset items: (path, label, isNight)
    for path, label, is_night in tqdm(items_test, desc='RNN attacks (backend-equiv)', unit='img'):
        try:
            # Backend-equivalent preprocessing: BGR [0,1], fixed 50x50
            bgr = preprocess_bgr_50x50(path)  # BGR [0,1]
        except Exception as e:
            print("[WARN] cannot load:", path, "->", e)
            continue

        # Add batch dimension for the model call
        x = bgr[np.newaxis, ...]
        needs_aux = bool(models.get('RNN_needs_aux'))
        if needs_aux:
            # Auxiliary input flows through unchanged (e.g., isNight)
            aux = np.array([[float(is_night)]], dtype=np.float32)

        # Clean prediction (baseline) 
        if needs_aux:
            clean_pred = rnn.predict([x, aux], verbose=0)[0]
        else:
            clean_pred = rnn.predict(x, verbose=0)[0]
        clean_idx = int(np.argmax(clean_pred))
        clean_correct += int(clean_idx == label)

        # FGSM (untargeted) on pixels only 
        if needs_aux:
            x_adv_fgsm = fgsm_pixels_multi(rnn, x, aux, eps=EPS_FGSM, from_logits=RNN_USE_LOGITS)
            adv_pred_fgsm = rnn.predict([x_adv_fgsm, aux], verbose=0)[0]
        else:
            x_adv_fgsm = fgsm_pixels(rnn, x, eps=EPS_FGSM, from_logits=RNN_USE_LOGITS)
            adv_pred_fgsm = rnn.predict(x_adv_fgsm, verbose=0)[0]

        # Save FGSM image: convert BGR→RGB for PNG encoding
        rgb_png = cv2.cvtColor((x_adv_fgsm[0]*255).astype('uint8'), cv2.COLOR_BGR2RGB)
        Image.fromarray(rgb_png).save(mirror_out_path(out_root_fgsm, path))

        idx_fgsm = int(np.argmax(adv_pred_fgsm))
        adv_fgsm_correct += int(idx_fgsm == label)

        # PGD (untargeted) on pixels only
        if needs_aux:
            x_adv_pgd = pgd_pixels_multi(rnn, x, aux, eps=EPS_PGD, step=PGD_STEP_SIZE, steps=PGD_STEPS, from_logits=RNN_USE_LOGITS)
            adv_pred_pgd = rnn.predict([x_adv_pgd, aux], verbose=0)[0]
        else:
            x_adv_pgd = pgd_pixels(rnn, x, eps=EPS_PGD, step=PGD_STEP_SIZE, steps=PGD_STEPS, from_logits=RNN_USE_LOGITS)
            adv_pred_pgd = rnn.predict(x_adv_pgd, verbose=0)[0]

        # Save PGD image: convert BGR→RGB for PNG encoding
        rgb_png = cv2.cvtColor((x_adv_pgd[0]*255).astype('uint8'), cv2.COLOR_BGR2RGB)
        Image.fromarray(rgb_png).save(mirror_out_path(out_root_pgd, path))

        idx_pgd = int(np.argmax(adv_pred_pgd))
        adv_pgd_correct += int(idx_pgd == label)

        total += 1
        
    # Accuracy summary (clean vs adversarial)
    print(f"[RNN] clean_acc={clean_correct/max(1,total):.4f}  fgsm_acc={adv_fgsm_correct/max(1,total):.4f}  pgd_acc={adv_pgd_correct/max(1,total):.4f}")
else:
    print("Skipping RNN.")

RNN attacks (backend-equiv):   0%|          | 5/14433 [00:43<34:01:35,  8.49s/img]

15. Sanity check — count saved adversarial PNG files

In [ ]:
# Count PNG files under each adversarial output subfolder and print a small summary.

def count_pngs(root):
    """Return number of .png files under `root` (recursively). If folder missing, return 0."""
    return len(glob.glob(os.path.join(root, '**', '*.png'), recursive=True))

for sub in ['ANN_fgsm','ANN_pgd','CNN_fgsm','CNN_pgd','RNN_fgsm','RNN_pgd']:
    p = os.path.join(SAVE_DIR_ROOT, sub)
    print(sub, count_pngs(p), "files")

ANN_fgsm 51826 files
ANN_pgd 51826 files
CNN_fgsm 51826 files
CNN_pgd 51826 files
RNN_fgsm 11599 files
RNN_pgd 11599 files
